<a href="https://colab.research.google.com/github/Asalulzy/Emotion-Recognition-from-Social-Media-Videos/blob/main/Satria_Data_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import noisereduce as nr
from scipy import signal
from sklearn.utils.class_weight import compute_class_weight
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift

In [ ]:
audio_dir = "C:/Satria_data/audio/audio/"
label_path = "C:/Satria_data/DATA/datatrain_standard.csv"

In [ ]:
# Load label
df = pd.read_csv(label_path)
print("Jumlah data awal:", len(df))
df.head()

In [ ]:
# Cek file yang ada di folder audio
extension = ".wav"
audio_files = set(os.listdir(audio_dir))
# Cek apakah semua audio ada di folder
df["exists"] = df["id"].apply(lambda x: f"{x}{extension}" in audio_files)
# Hitung yang missing
missing_count = len(df) - df["exists"].sum()
print(f"\nFile audio yang tidak ditemukan: {missing_count}")

In [ ]:
# Hapus data yang tidak ada audionya
df_clean = df[df["exists"]].copy().reset_index(drop=True)
df_clean.drop(columns=["exists"], inplace=True)

print("Jumlah data setelah dibersihkan:", len(df_clean))
df_clean.head()

In [ ]:
# Mapping manual
emotion_map = {
    "Proud": 0,
    "Trust": 1,
    "Joy": 2,
    "Surprise": 3,
    "Neutral": 4,
    "Sadness": 5,
    "Fear": 6,
    "Anger": 7
}

# Apply mapping ke dataframe
df_clean["emotion_encoded"] = df_clean["emotion_standard"].map(emotion_map)
num_classes = len(emotion_map)

print(df_clean[["id", "emotion_standard", "emotion_encoded"]].head())

In [ ]:
#Tambahkan kolom path ke file audio
df_clean = df_clean.drop(columns=["video", "emotion"])
df_clean["audio"] = df_clean["id"].astype(str).apply(lambda x: os.path.join("C:/Satria_data/audio/audio/", f"{x}.wav"))
# 3. Cek data
print("Jumlah data:", len(df_clean))
df_clean.head()

In [ ]:
import pandas as pd
from IPython.display import Audio, display

# misalnya clean_df sudah ada
# cek 5 audio pertama
for path in df_clean['audio'].head(5):
    display(Audio(filename=path))

In [ ]:
# Folder untuk menyimpan audio yang sudah dibersihkan
cleaned_dir = "C:/Satria_data/audio/cleaned_audio/"
os.makedirs(cleaned_dir, exist_ok=True)

def remove_noise_music(audio, sr):
    """Hilangkan noise + musik latar"""
    try:
        # Noise reduction
        reduced_noise = nr.reduce_noise(
            y=audio,
            sr=sr,
            prop_decrease=0.8,
            n_fft=1024,
            win_length=512
        )
        # High-pass filter untuk mengurangi musik dramatis
        nyquist = sr / 2
        high_pass_freq = 100
        b, a = signal.butter(4, high_pass_freq/nyquist, btype='high')
        filtered_audio = signal.filtfilt(b, a, reduced_noise)
        return filtered_audio
    except Exception as e:
        print(f"Noise removal error: {e}")
        return audio

# Proses semua audio di df_clean
cleaned_rows = []
for idx, row in df_clean.iterrows():
    file_path = row["audio"]
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    try:
        audio, sr = librosa.load(file_path, sr=22050)
        clean_audio = remove_noise_music(audio, sr)

        clean_file = f"{file_name}_clean.wav"
        clean_path = os.path.join(cleaned_dir, clean_file)
        sf.write(clean_path, clean_audio, sr)

        new_row = row.copy()
        new_row["audio"] = clean_path
        cleaned_rows.append(new_row)
        print(f"✓ Cleaned: {file_name}")
    except Exception as e:
        print(f"✗ Failed: {file_path} | {e}")

# Buat dataframe hasil bersih
df_cleaned = pd.DataFrame(cleaned_rows)
print("\nJumlah data setelah dibersihkan:", len(df_cleaned))
df_cleaned.head()

In [ ]:
# Cek 5 audio hasil bersih
for path in df_cleaned['audio'].head(5):
    display(Audio(filename=path))

In [ ]:
# Folder output untuk potongan audio yang SUDAH BERSIH
chunks_dir = "C:/Satria_data/audio/chunks_cleaned/"
os.makedirs(chunks_dir, exist_ok=True)

def chunk_audio(file_path, output_dir, sample_rate=22050, window_duration=10, overlap=5):
    y, sr = librosa.load(file_path, sr=sample_rate)
    win = int(window_duration * sr)
    step = int((window_duration - overlap) * sr)
    file_name = os.path.splitext(os.path.basename(file_path))[0]

    paths = []
    for i, start in enumerate(range(0, len(y) - win + 1, step)):
        end = start + win
        chunk = y[start:end]
        out_path = os.path.join(output_dir, f"{file_name}_chunk{i}.wav")
        sf.write(out_path, chunk, sr)
        paths.append(out_path)
    return paths

rows = []
for _, row in df_cleaned.iterrows():  # df_cleaned: hasil cleaning kamu
    try:
        chunk_paths = chunk_audio(row["audio"], chunks_dir, sample_rate=22050, window_duration=10, overlap=5)
        for i, cp in enumerate(chunk_paths):
            new_row = row.copy()
            new_row["chunk_id"] = i
            new_row["audio"] = cp
            rows.append(new_row)
    except Exception as e:
        print("Gagal chunk:", row["audio"], "|", e)

df_chunks = pd.DataFrame(rows).reset_index(drop=True)
print("Jumlah baris df_chunks:", len(df_chunks))
df_chunks.head()

In [ ]:
import numpy as np
import librosa
import pandas as pd
import time

def feature_row_from_path(path, sr=22050, n_mfcc=13):
    y, sr = librosa.load(path, sr=sr)

    if y.size == 0:
        raise ValueError("File audio kosong atau tidak valid")

    # Time-domain
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    # Frequency / Perceptual
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    def m_s(a):
        return np.mean(a), np.std(a)

    feats = {}
    feats["zcr_mean"], feats["zcr_std"] = m_s(zcr)
    feats["rms_mean"], feats["rms_std"] = m_s(rms)
    feats["centroid_mean"], feats["centroid_std"] = m_s(spec_centroid)
    feats["bandwidth_mean"], feats["bandwidth_std"] = m_s(spec_bw)
    feats["rolloff_mean"], feats["rolloff_std"] = m_s(spec_roll)

    for i in range(contrast.shape[0]):
        feats[f"contrast{i+1}_mean"] = np.mean(contrast[i])
        feats[f"contrast{i+1}_std"]  = np.std(contrast[i])

    for i in range(chroma.shape[0]):
        feats[f"chroma{i+1}_mean"] = np.mean(chroma[i])
        feats[f"chroma{i+1}_std"]  = np.std(chroma[i])

    for i in range(tonnetz.shape[0]):
        feats[f"tonnetz{i+1}_mean"] = np.mean(tonnetz[i])
        feats[f"tonnetz{i+1}_std"]  = np.std(tonnetz[i])

    for i in range(mfcc.shape[0]):
        feats[f"mfcc{i+1}_mean"] = np.mean(mfcc[i])
        feats[f"mfcc{i+1}_std"]  = np.std(mfcc[i])

    return feats


# --- Ekstraksi fitur dengan progress log + ETA ---
feature_rows = []
base_df = df_chunks if 'df_chunks' in globals() else df_cleaned
total = len(base_df)
start_time = time.time()

for i, row in base_df.iterrows():
    try:
        feats = feature_row_from_path(row["audio"], sr=22050, n_mfcc=13)
        out = {**row.to_dict(), **feats}
        feature_rows.append(out)

        # Progress
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)  # rata-rata waktu per file
        remaining = avg_time * (total - (i + 1))
        print(f"▶️ Proses {i+1}/{total} | ID: {row['id']} | ETA: {remaining/60:.2f} menit")

    except Exception as e:
        print(f"✗ Gagal ekstraksi fitur: {row['audio']} | {e}")

elapsed = time.time() - start_time
print(f"\n✅ Ekstraksi selesai dalam {elapsed/60:.2f} menit")

feature_df = pd.DataFrame(feature_rows)
print("feature_df shape:", feature_df.shape)
feature_df.head()

In [ ]:
# Simpan ke CSV
feature_df.to_csv(r'C:\Satria_data\DATA\feature_df.csv', index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import numpy as np

# Tentukan fitur kolom (semua kolom numerik fitur, exclude meta)
meta_cols = ["id", "audio", "emotion", "emotion_standard", "emotion_encoded", "video", "chunk_id"]
feature_cols = [c for c in feature_df.columns if c not in meta_cols and feature_df[c].dtype != "O"]

X = feature_df[feature_cols].values
y = feature_df["emotion_encoded"].values  # sudah kamu siapkan dari mapping

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Model 1: Random Forest (baseline paper)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)
print("RandomForest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

# Model 2: Gradient Boosting (baseline paper)
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_s, y_train)
gb_pred = gb.predict(X_test_s)
print("GradientBoosting Accuracy:", accuracy_score(y_test, gb_pred))
print(classification_report(y_test, gb_pred))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

# Tentukan fitur kolom (semua kolom numerik fitur, exclude meta)
meta_cols = ["id", "audio", "emotion", "emotion_standard", "emotion_encoded", "video", "chunk_id"]
feature_cols = [c for c in feature_df.columns if c not in meta_cols and feature_df[c].dtype != "O"]

X = feature_df[feature_cols].values
y = feature_df["emotion_encoded"].values  # sudah kamu siapkan dari mapping

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardisasi fitur
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# =========================
# Model 1: Random Forest dengan class_weight
# =========================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'  # menyesuaikan bobot tiap kelas otomatis
)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)
print("RandomForest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

# =========================
# Model 2: Gradient Boosting dengan sample_weight
# =========================
# GradientBoosting tidak memiliki class_weight langsung, jadi gunakan sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_s, y_train, sample_weight=sample_weights)
gb_pred = gb.predict(X_test_s)
print("GradientBoosting Accuracy:", accuracy_score(y_test, gb_pred))
print(classification_report(y_test, gb_pred))
